In [3]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("ChurnPipeline")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .getOrCreate())
print(spark.version)

4.2.0


In [4]:
def to_bronze(path, name, batch_id="batch_001"):
    df = (spark.read.option("header", True)
          .option("inferSchema", False)      # keep everything as string (raw)
          .csv(path))
    df = (df.withColumn("_source_file", F.lit(path))
            .withColumn("_ingest_ts", F.current_timestamp())
            .withColumn("_batch_id", F.lit(batch_id)))
    df.write.mode("overwrite").parquet(f"../bronze/{name}")
    print(name, df.count(), "rows")
    return df

bronze_cust = to_bronze("../data/raw/telco.csv", "customers")
bronze_tix  = to_bronze("../data/raw/tickets.csv", "tickets")

spark.read.parquet("../bronze/customers").printSchema()
spark.read.parquet("../bronze/customers").show(5, truncate=False)

customers 7043 rows
tickets 15000 rows
root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: string (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: string (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: string (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)
 |-- _source_file: string (nullable = true)
 